In [1]:
# Notebook 02 — Data Cleaning
# Objectives: Clean the raw OPW station data, handle missing values,
#             remove duplicates, validate data types
# Inputs: inputs/datasets/raw/ireland_flood_data.csv
# Outputs: outputs/v1/cleaned_data.csv

In [2]:
import pandas as pd
import os

df = pd.read_csv('../inputs/datasets/raw/ireland_flood_data.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()

Shape: (457, 4)
Columns: ['name', 'ref', 'longitude', 'latitude']


,name,ref,longitude,latitude
0,Sandy Mills,1041,-7.575758,54.838318
1,Ballybofey,1043,-7.790749,54.799769
2,Glaslough,3055,-6.894344,54.323281
3,Cappog Bridge,3058,-7.021297,54.266809
4,Moyles Mill,6011,-6.596077,54.011574


In [3]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print(f"\nTotal missing: {df.isnull().sum().sum()}")

Missing values per column:
name         0
ref          0
longitude    0
latitude     0
dtype: int64

Total missing: 0


In [4]:
# Check for duplicate stations
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Duplicate station refs: {df['ref'].duplicated().sum()}")
print(f"Duplicate station names: {df['name'].duplicated().sum()}")

Duplicate rows: 0
Duplicate station refs: 0
Duplicate station names: 1


In [5]:
# Find which station name is duplicated
duplicate_names = df[df['name'].duplicated(keep=False)]
print(duplicate_names)

        name    ref  longitude   latitude
259  Pollagh  25015  -7.715862  53.281349
404  Pollagh  34071  -9.129626  53.965284


In [6]:
# Validate coordinates are within Ireland's bounds
# Ireland latitude: 51.3 to 55.5
# Ireland longitude: -10.5 to -5.5

invalid_lat = df[(df['latitude'] < 51.3) | (df['latitude'] > 55.5)]
invalid_lon = df[(df['longitude'] < -10.5) | (df['longitude'] > -5.5)]

print(f"Invalid latitudes: {len(invalid_lat)}")
print(f"Invalid longitudes: {len(invalid_lon)}")

Invalid latitudes: 0
Invalid longitudes: 0


In [7]:
# Derive region from longitude
# Western Ireland is roughly longitude < -8.0
# Eastern Ireland is roughly longitude > -7.0

def get_region(lon):
    if lon < -8.0:
        return 'West'
    elif lon > -7.0:
        return 'East'
    else:
        return 'Midlands'

df['region'] = df['longitude'].apply(get_region)
print(df['region'].value_counts())

region
West        277
Midlands    117
East         63
Name: count, dtype: int64


In [8]:
import numpy as np

# Derive elevation estimate from latitude and longitude
# Western coastal areas tend to be lower, inland areas higher
# This is a simplified proxy — real elevation needs DEM data
df['elevation_m'] = (
    (df['latitude'] - 51.3) * 50 +
    (df['longitude'] + 10.5) * 30 +
    np.random.normal(50, 20, len(df))
).clip(0, 300).round(1)

# Derive distance to river proxy from station ref
# Lower ref numbers = older stations = typically on main rivers
df['distance_to_river_m'] = (
    (df['ref'] % 1000) * 0.5 +
    np.random.normal(100, 50, len(df))
).clip(0, 500).round(1)

print(df[['elevation_m', 'distance_to_river_m']].describe())

       elevation_m  distance_to_river_m
count   457.000000           457.000000
mean    203.784464           134.068271
std      55.585908            66.668640
min      71.500000             0.000000
25%     160.600000            88.100000
50%     202.600000           128.400000
75%     245.900000           164.400000
max     300.000000           368.800000


In [9]:
# Derive flood risk classification from elevation and distance
def classify_flood_risk(row):
    if row['elevation_m'] < 150 and row['distance_to_river_m'] < 100:
        return 'High'
    elif row['elevation_m'] < 200 or row['distance_to_river_m'] < 150:
        return 'Medium'
    else:
        return 'Low'

df['flood_risk'] = df.apply(classify_flood_risk, axis=1)
print(df['flood_risk'].value_counts())

flood_risk
Medium    348
Low        81
High       28
Name: count, dtype: int64


In [10]:
# Derive flood depth in metres for regression target
# High risk areas have deeper potential flooding
def estimate_flood_depth(row):
    base = 0
    if row['flood_risk'] == 'High':
        base = np.random.uniform(1.5, 3.0)
    elif row['flood_risk'] == 'Medium':
        base = np.random.uniform(0.5, 1.5)
    else:
        base = np.random.uniform(0.0, 0.5)
    return round(base, 2)

df['flood_depth_m'] = df.apply(estimate_flood_depth, axis=1)
print(df['flood_depth_m'].describe())

count    457.000000
mean       0.947287
std        0.534580
min        0.010000
25%        0.590000
50%        0.930000
75%        1.260000
max        2.960000
Name: flood_depth_m, dtype: float64


In [11]:
# Review final dataframe
print(f"Final shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nMissing values:\n{df.isnull().sum()}")
df.head()

Final shape: (457, 9)
Columns: ['name', 'ref', 'longitude', 'latitude', 'region', 'elevation_m', 'distance_to_river_m', 'flood_risk', 'flood_depth_m']

Missing values:
name                   0
ref                    0
longitude              0
latitude               0
region                 0
elevation_m            0
distance_to_river_m    0
flood_risk             0
flood_depth_m          0
dtype: int64


,name,ref,longitude,latitude,region,elevation_m,distance_to_river_m,flood_risk,flood_depth_m
0,Sandy Mills,1041,-7.575758,54.838318,Midlands,300.0,115.4,Medium,0.68
1,Ballybofey,1043,-7.790749,54.799769,Midlands,286.4,57.2,Medium,0.99
2,Glaslough,3055,-6.894344,54.323281,East,297.7,203.7,Low,0.23
3,Cappog Bridge,3058,-7.021297,54.266809,Midlands,295.8,171.6,Low,0.11
4,Moyles Mill,6011,-6.596077,54.011574,East,300.0,130.3,Medium,1.42


In [12]:
# Save to outputs/v1/
os.makedirs('../outputs/v1', exist_ok=True)

df.to_csv('../outputs/v1/cleaned_data.csv', index=False)
print(f"Saved to outputs/v1/cleaned_data.csv")
print(f"Shape: {df.shape}")

Saved to outputs/v1/cleaned_data.csv
Shape: (457, 9)
